# Method-preserving reference benchmark on Kaggle

This notebook runs the FUnIE-GAN, UColor, project U-Net, Water-Net, and UWFormer reference reruns on UIEB and LSUI. It standardizes the data, split, external budget, validation selection, evaluator, and reporting while preserving each method's defining inputs and training recipe. It does **not** run the physics ablation or modern-architecture benchmark.

Recommended workflow: run the complete smoke matrix first, save its outputs, then run the full benchmark in method/seed shards. Kaggle sessions are time-limited, so preserve `/kaggle/working/reference_outputs` as a private Kaggle Dataset between sessions.

## 1. Configuration
Edit the two attached-dataset paths to match the names shown under `/kaggle/input`. Execution flags default to false to avoid starting training accidentally.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/heniath/underwater-image-enhancement.git'
REPO_BRANCH = 'learnable-physics-extractor'
REPO_DIR = Path('/kaggle/working/underwater-image-enhancement')

# Change these to the roots of the Kaggle Datasets you attached.
UIEB_INPUT = Path('/kaggle/input/uieb-dataset')
LSUI_INPUT = Path('/kaggle/input/lsui-dataset')
DATA_ROOT = Path('/kaggle/working/reference_data')
OUTPUT_ROOT = Path('/kaggle/working/reference_outputs')
TORCH_CACHE = Path('/kaggle/working/torch_cache')

RUN_TESTS = True
RUN_SMOKE = False
RUN_FULL = False
RUN_EFFICIENCY = False
USE_RAM_CACHE = True

# Full-run shard. Keep canonical method identifiers.
FULL_METHODS = ['funie_gan']
FULL_DATASETS = ['UIEB', 'LSUI']
FULL_SEEDS = [0]

print('Configured. Set RUN_SMOKE=True only after attaching both datasets.')

## 2. GPU and Kaggle environment

In [ ]:
import os, platform, subprocess, sys

print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)
if not Path('/kaggle').exists():
    raise RuntimeError('This notebook is configured for the Kaggle runtime.')

## 3. Clone/update and install
Internet must be enabled for the initial Git clone, dependency installation, and first VGG16/VGG19 weight download. Later sessions may reuse attached code and cached weights.

In [ ]:
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[dev,profile,visualization]'], check=True)
os.chdir(REPO_DIR)

# Editable installs write a .pth file that an already-running Kaggle kernel may
# not process until restart. Activate the source tree in this kernel explicitly.
import importlib
repo_src = str(REPO_DIR / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import uwir

print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('uwir imported from:', Path(uwir.__file__).resolve())

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle notebook settings before training.')

## 4. Persistent auxiliary-weight cache

In [ ]:
TORCH_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['UWIR_TORCH_HOME'] = str(TORCH_CACHE)
print('VGG/cache directory:', TORCH_CACHE)

## 5. Resolve attached UIEB and LSUI datasets
Kaggle inputs are read-only. This cell creates writable-directory symlinks without copying image data.

In [ ]:
def find_uieb_root(root: Path) -> Path:
    candidates = [root] + [p.parent for p in root.rglob('raw-890')]
    for candidate in candidates:
        if (candidate / 'raw-890').is_dir() and (candidate / 'reference-890').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find raw-890/reference-890 below {root}')

def replace_symlink(link: Path, target: Path):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f'Refusing to replace non-symlink: {link}')
    link.symlink_to(target, target_is_directory=True)

if not UIEB_INPUT.is_dir():
    raise FileNotFoundError(f'Attach UIEB or change UIEB_INPUT: {UIEB_INPUT}')
if not LSUI_INPUT.is_dir():
    raise FileNotFoundError(f'Attach LSUI or change LSUI_INPUT: {LSUI_INPUT}')

DATA_ROOT.mkdir(parents=True, exist_ok=True)
replace_symlink(DATA_ROOT / 'UIEB', find_uieb_root(UIEB_INPUT))
replace_symlink(DATA_ROOT / 'LSUI', LSUI_INPUT.resolve())
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('UIEB ->', (DATA_ROOT / 'UIEB').resolve())
print('LSUI ->', (DATA_ROOT / 'LSUI').resolve())

## 6. Dataset discovery and strict pairing preflight

In [ ]:
from uwir.datasets.lsui import compact_tree, discover_lsui, prepare_lsui_splits
from uwir.datasets.uieb import discover_uieb, prepare_uieb_splits

print('LSUI compact tree:')
print(compact_tree(DATA_ROOT / 'LSUI'))
uieb_pairs = discover_uieb(DATA_ROOT / 'UIEB')
lsui_pairs, lsui_report = discover_lsui(DATA_ROOT / 'LSUI')
print('UIEB pairs:', len(uieb_pairs))
print('LSUI report:', lsui_report)
uieb_splits = prepare_uieb_splits(DATA_ROOT / 'UIEB', OUTPUT_ROOT / 'splits/uieb_split_manifest.json')
lsui_splits = prepare_lsui_splits(DATA_ROOT / 'LSUI', OUTPUT_ROOT / 'splits/lsui_split_manifest.json')
print('UIEB:', {k: len(v) for k, v in uieb_splits.items()})
print('LSUI:', {k: len(v) for k, v in lsui_splits.items()})

## 7. Code tests and method configuration

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

In [ ]:
import pandas as pd
from uwir.reference_methods import REFERENCE_METHODS

rows = []
for name, cls in REFERENCE_METHODS.items():
    cfg = cls.training_config()
    rows.append({'method': name, 'inputs': cfg['input_formulation'], 'losses': cfg['losses'], 'optimizers': cfg['optimizers'], 'schedulers': cfg['schedulers']})
pd.DataFrame(rows)

## 8. Benchmark command helper

In [ ]:
def run_benchmark(mode, methods=None, datasets=None, seeds=None, efficiency=False):
    command = [
        sys.executable, '-m', 'scripts.reference_methods_benchmark',
        f'--{mode}', '--device', 'cuda',
        '--data-root', str(DATA_ROOT), '--output-root', str(OUTPUT_ROOT),
    ]
    if not USE_RAM_CACHE:
        command.append('--no-ram-cache')
    if methods:
        command.extend(['--methods', *methods])
    if datasets:
        command.extend(['--datasets', *datasets])
    if seeds is not None:
        command.extend(['--seeds', *map(str, seeds)])
    if efficiency:
        command.append('--efficiency')
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)

## 9. Required ten-combination smoke matrix
Set `RUN_SMOKE=True` in the configuration cell and rerun from there. A forward-only test is not accepted: the CLI performs training updates, validation, checkpoint reload, test metrics, and CSV writing.

In [ ]:
if RUN_SMOKE:
    run_benchmark('smoke')
else:
    print('Smoke disabled. Set RUN_SMOKE=True when ready.')

In [ ]:
smoke_csv = OUTPUT_ROOT / 'smoke_results.csv'
if smoke_csv.exists():
    smoke_results = pd.read_csv(smoke_csv)
    display(smoke_results.sort_values(['dataset', 'method']))
    print('Combinations:', len(smoke_results), '/ 10')
else:
    print('No smoke_results.csv yet.')

## 10. Resumable full-run shard
Full mode is blocked until all ten smoke combinations pass. Choose a small shard in the configuration cell. Repeating a shard resumes incomplete checkpoints and skips completed runs.

In [ ]:
print('Full shard:', {'methods': FULL_METHODS, 'datasets': FULL_DATASETS, 'seeds': FULL_SEEDS})
if RUN_FULL:
    run_benchmark('full', methods=FULL_METHODS, datasets=FULL_DATASETS, seeds=FULL_SEEDS)
else:
    print('Full training disabled. Set RUN_FULL=True after smoke passes.')

## 11. Efficiency benchmark
Efficiency is measured once per method at batch size one and 256×256. FUnIE-GAN excludes its discriminator.

In [ ]:
if RUN_EFFICIENCY:
    run_benchmark('full', methods=FULL_METHODS, datasets=FULL_DATASETS, seeds=FULL_SEEDS, efficiency=True)
else:
    print('Efficiency benchmark disabled.')

## 12. Progress and results

In [ ]:
completed = list(OUTPUT_ROOT.glob('*/*/seed_*/test_metrics.json'))
checkpoints = list(OUTPUT_ROOT.glob('*/*/seed_*/last_model.pth'))
print('Completed full runs:', len(completed), '/ 30')
print('Runs with resumable last checkpoints:', len(checkpoints))
for filename in ('per_run_results.csv', 'aggregate_results.csv', 'efficiency_results.csv'):
    path = OUTPUT_ROOT / filename
    if path.exists():
        print('\n', filename)
        display(pd.read_csv(path))

## 13. Save artifacts between Kaggle sessions
Use **Save Version** so `/kaggle/working/reference_outputs` is retained in notebook output, or publish that directory as a private Kaggle Dataset. In a later session, copy the previous output dataset into the writable path before running any benchmark cell.

In [ ]:
# Example restore command for a later session; edit the attached dataset name.
# import shutil
# previous = Path('/kaggle/input/reference-benchmark-checkpoints/reference_outputs')
# if previous.exists() and not OUTPUT_ROOT.exists():
#     shutil.copytree(previous, OUTPUT_ROOT)

print('Persistent artifacts to save:', OUTPUT_ROOT)
print('Auxiliary-weight cache to save for internet-free reuse:', TORCH_CACHE)